# 06 · OD 분석(상반기) — 2026H1 기간 정합 집계

`05_OD분석.ipynb`(2026-07-22~28 7일치)의 **기간 정합 확장판**. 7월 1주 표본은 계절·방학·휴가철 편차를 담지 못하고 지방 승하차 데이터(부산·대구·대전·광주, 2026-01~06)와 기간이 어긋나므로, 서울 역간 OD 월별 아카이브 6개(2026-01~06, 181일)를 같은 방식으로 집계해 지방 데이터와 **완전히 중첩되는 상반기 산출물**(`*_2026H1.csv`)을 만든다.

`od_h1_analysis.py` 의 함수를 그대로 호출한다 (05와 동일 컨벤션). 05 노트북·`od_analysis.py`·기존 산출물은 변경하지 않는다.

- 입력: `data/od/kscc_dx_ra_od_2026MM.zip` 6개 **또는** 중간 캐시 `data/od/h1_cache/` (배치 안내: `data/od/README.md`)
- 기간 정합성: 부산·대구·대전·광주 승하차와 **완전 중첩**, 인천 1·2호선 승하차만 2026-01~04 **부분 중첩**
- 서울 OD 원천 결측 2일(2026-01-09, 05-16 — 포털 zip에 헤더만 있는 CSV)은 평균 분모에서 제외 → 유효 179일

In [1]:
# 저장소 루트에서 실행되도록 경로 이동 (notebooks/ 안에서 열었을 때 대비)
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

## 1. 상반기 181일 집계 → 노드쌍 통행량

역사_ID ↔ node_id 매핑은 05 산출물(`od_station_mapping.csv`, 커밋됨)을 재사용한다. 일별 집계는 `data/od/h1_cache/` 에 캐시 — **캐시가 있으면 2~3분, 월별 zip 파싱 시 15~20분** 소요.

In [2]:
import pandas as pd
import od_analysis as oda
import od_h1_analysis as odh

nodes = oda.load_nodes()
mapping = pd.read_csv('data/od/processed/od_station_mapping.csv',
                      dtype={'역사_ID': str})
print('그래프 노드', len(nodes), '/ 매핑 역사_ID', len(mapping))

acc_all, acc_wd, acc_we, stats = odh.aggregate_h1(mapping)
daily_avg, peak, meta = odh.build_h1_outputs(acc_all, acc_wd, acc_we, stats)

그래프 노드 1094 / 매핑 역사_ID 774


[od]   9/181일 (20260109) — **원천 결측(헤더만)** -> 제외


[od]  20/181일 (20260120, cache) — 경과 10s, 누적쌍 411,427


[od]  40/181일 (20260209, cache) — 경과 20s, 누적쌍 433,164


[od]  60/181일 (20260301, cache) — 경과 30s, 누적쌍 445,296


...

[od] 181/181일 (20260630, cache) — 경과 93s, 누적쌍 472,706


[od] 유효 179일 (주중 128 / 주말 51) — 원천 결측 2일: ['20260109', '20260516']


[od] 상반기 총 통행 1,145,489,156 -> 매핑 후 1,131,623,377 (보존율 98.79%)


[out] od_daily_avg_2026H1.csv: 472,706 노드쌍


[out] od_peak_2026H1.csv: 419,712 노드쌍 (주중 128일 평균)


### 상위 통행쌍 — 상식 부합 점검 (7월 1주와 동일하게 강남·잠실 축이 최상위여야 정상)

In [3]:
lab = nodes.set_index('node_id')
top = daily_avg[daily_avg.node_o != daily_avg.node_d].head(10).copy()
top['출발'] = top.node_o.map(lab.역사명) + ' (' + top.node_o.map(lab.노선명) + ')'
top['도착'] = top.node_d.map(lab.역사명) + ' (' + top.node_d.map(lab.노선명) + ')'
top[['출발', '도착', 'trips_avg_daily', 'trips_avg_weekday', 'trips_avg_weekend']]

,출발,도착,trips_avg_daily,trips_avg_weekday,trips_avg_weekend
1,강남 (2호선),잠실(송파구청) (2호선),5098.18,5563.81,3929.55
2,잠실(송파구청) (2호선),강남 (2호선),4974.35,5357.37,4013.04
3,을지로입구 (2호선),홍대입구 (2호선),3698.49,3681.79,3740.39
4,강남 (2호선),신림 (2호선),3604.89,4140.91,2259.59
5,홍대입구 (2호선),을지로입구 (2호선),3563.88,3498.77,3727.31
6,신림 (2호선),강남 (2호선),3462.46,3921.32,2310.80
7,삼성(무역센터) (2호선),잠실(송파구청) (2호선),3456.32,3689.96,2869.94
8,서울대입구(관악구청) (2호선),강남 (2호선),3291.97,3743.64,2158.35
9,강남 (2호선),서울대입구(관악구청) (2호선),3239.54,3706.88,2066.61
10,잠실(송파구청) (2호선),삼성(무역센터) (2호선),3157.62,3454.70,2412.02


## 2. 노드 가중치 — 서울 OD + 지방 승하차 (기간 정합 상태 표시)

`weight_source`: `seoul_od_2026H1`(완전 중첩) / `boarding_data_2026H1`(부산·대구·대전·**광주 신규**, 완전 중첩) / `incheon_boarding_202601-04_partial`(인천 1·2호선, 운영기관 파일 우선 — **부분 중첩**) / `none`.

In [4]:
reg, reg_inc, miss = odh.regional_weights_h1(nodes)
weights = odh.build_node_weights_h1(nodes, daily_avg, mapping, reg, reg_inc)
weights.weight_source.value_counts().to_frame('노드 수')

[incheon] 2026-01~04 필터: 14,400행, 2026-01-01 ~ 2026-04-30, 1·2호선 59개 역


[weights] 부산·대구·대전·광주 248개 / 인천 1·2호선 60개 노드


[out] node_weights_2026H1.csv: 1094 노드, 커버리지 {'seoul_od_2026H1': 703, 'boarding_data_2026H1': 248, 'none': 83, 'incheon_boarding_202601-04_partial': 60}


,노드 수
weight_source,
seoul_od_2026H1,703
boarding_data_2026H1,248
none,83
incheon_boarding_202601-04_partial,60


### 광주 1호선 승차 상위 — 신규 반영 확인 (05에서는 전부 `none` 이었음)

In [5]:
weights[weights.region == '광주'].nlargest(
    10, 'boarding_daily_avg')[['역사명', '노선명',
                               'boarding_daily_avg', 'alighting_daily_avg']]

,역사명,노선명,boarding_daily_avg,alighting_daily_avg
62,광주송정역,광주도시철도 1호선,5245.3,4952.8
58,상무,광주도시철도 1호선,4049.9,4231.3
48,남광주,광주도시철도 1호선,4019.2,4179.9
50,금남로4가,광주도시철도 1호선,3999.2,3959.4
57,운천,광주도시철도 1호선,3291.1,3238.5
49,문화전당(구도청),광주도시철도 1호선,3208.1,3275.5
54,농성,광주도시철도 1호선,2930.1,2753.2
61,송정공원,광주도시철도 1호선,2763.5,2760.1
56,쌍촌,광주도시철도 1호선,2608.5,2087.5
52,양동시장,광주도시철도 1호선,2346.9,2324.6


## 3. 검증 — 7월 1주(05 산출물) 대비 순위 상관 (Spearman)

기간을 바꿔도 노드 중요도·통행 구조 순위가 유지되는지(기간 강건성) 확인. 수도권 ρ가 1에 가까우면 7월 1주 기준 분석 결과가 상반기 기준으로도 유지됨을 뜻한다.

In [6]:
corrs = odh.validate_vs_july(daily_avg, weights)
pd.DataFrame(corrs).T

[spearman] 수도권(seoul_od, 동일 원천): n=703 승차 0.9989 / 하차 0.9988


[spearman] 인천 1·2호선(H1=운영기관 vs 7월=seoul_od, 원천 상이): n=60 승차 0.7867 / 하차 0.7644


[spearman] 전체(가중치 보유 공통 노드): n=991 승차 0.9498 / 하차 0.9498


[spearman] OD 노드쌍(공통쌍 통행량): n=380,228 승차 0.9722


,n,boarding,alighting
"수도권(seoul_od, 동일 원천)",703.0,0.998927,0.998848
"인천 1·2호선(H1=운영기관 vs 7월=seoul_od, 원천 상이)",60.0,0.786719,0.764435
전체(가중치 보유 공통 노드),991.0,0.949830,0.949839
OD 노드쌍(공통쌍 통행량),380228.0,0.972179,NaN


## 산출물 (data/od/processed/)

- `node_weights_2026H1.csv` — 커밋됨 (실행 없이 바로 사용 가능)
- `od_daily_avg_2026H1.csv` / `od_peak_2026H1.csv` — 용량 제외(.gitignore), `python od_h1_analysis.py` 실행으로 재생성
- 활용법(승객가중 효율·단절 통행량·첨두 취약성)은 `docs/README_OD분석.md` 의 analyze.py 연계 제안과 동일 — 기간 정합 분석에는 `*_2026H1.csv` 을, 민감도 분석에는 7월 1주 산출물을 권장